# Investigating MORE

Mackenzie Core Resources ETF, TSX: MORE. What does one ticker buy, what does it cost, and what risk does it carry? This notebook recomputes the performance tables in the report from the weekly price file, so every number can be checked.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

prices = pd.read_csv('data/weekly_prices_cad.csv', index_col=0, parse_dates=True)
prices.head()

In [ ]:
# Weekly returns. BLEND is a 50/50 XEG/XMA portfolio, rebalanced weekly.
rets = prices.pct_change().dropna()
rets['BLEND'] = 0.5 * rets['XEG'] + 0.5 * rets['XMA']
print('Weeks:', len(rets))
print('From', rets.index[0].date(), 'to', rets.index[-1].date())

In [ ]:
def stats(s, rf=0.02):
    ann_ret = s.mean() * 52
    vol = s.std() * np.sqrt(52)
    cum = (1 + s).cumprod()
    dd = cum / cum.cummax() - 1
    return {
        'annual return %': round(float(ann_ret * 100), 2),
        'volatility %': round(float(vol * 100), 2),
        'return / volatility': round(float(ann_ret / vol), 2),
        'Sharpe (2% risk free)': round(float((ann_ret - rf) / vol), 2),
        'max drawdown %': round(float(dd.min() * 100), 2),
        'worst week %': round(float(s.min() * 100), 2),
        'weekly CVaR 95% %': round(float(s[s <= s.quantile(0.05)].mean() * 100), 2),
        'total gain %': round(float((cum.iloc[-1] - 1) * 100), 2),
        '$10,000 becomes': round(float(10000 * cum.iloc[-1]), 2),
    }

metrics = pd.DataFrame({k: stats(rets[k]) for k in rets.columns}).T
metrics

In [ ]:
# Growth of $10,000, weekly, in CAD
growth = (1 + rets).cumprod() * 10000
growth.plot(figsize=(10, 5), title='Growth of $10,000 in CAD, weekly')
plt.ylabel('$')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# MORE drawdown: fall from the previous peak, in percent
dd = growth / growth.cummax() - 1
plt.figure(figsize=(10, 5))
plt.plot(dd.index, dd['MORE'] * 100, color='#1D5C30')
plt.fill_between(dd.index, dd['MORE'] * 100, 0, alpha=0.2, color='#1D5C30')
plt.title('MORE drawdown: fall from the previous peak, in percent')
plt.ylabel('%')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Total return by period
for label, a, b in [('2024 stub (20 Sep to 31 Dec)', '2024-09-13', '2024-12-31'),
                    ('2025', '2025-01-01', '2025-12-31'),
                    ('2026 to 13 Nov', '2026-01-01', '2026-11-13')]:
    x = rets.loc[a:b]
    print(label, {k: round(float(((1 + x[k]).prod() - 1) * 100), 2) for k in x.columns})

In [ ]:
# Correlation with MORE
rets.corr()['MORE'].round(3)

In [ ]:
# Monte Carlo: 10,000 paths over the next 52 weeks, from the last close.
# Two drifts: MORE's own average weekly return, and a conservative 7% a year.
rng = np.random.default_rng(42)
mu_w = rets['MORE'].mean()
sigma_w = rets['MORE'].std()
start = prices['MORE'].iloc[-1]
for label, mu in [('historical', mu_w), ('conservative 7% a year', 0.07 / 52)]:
    draws = rng.normal(mu, sigma_w, size=(10000, 52))
    paths = start * np.cumprod(1 + draws, axis=1)
    terminal = paths[:, -1]
    print(label, '| median $', round(float(np.median(terminal)), 2),
          '| 5th $', round(float(np.percentile(terminal, 5)), 2),
          '| 95th $', round(float(np.percentile(terminal, 95)), 2),
          '| chance of a loss', round(float((terminal < start).mean() * 100), 1), '%')

In [ ]:
# Commodity scenarios: MORE weekly return on WTI oil, gold, and XIC.
fac = pd.read_csv('data/factor_weekly_returns.csv', index_col=0, parse_dates=True)
df = pd.concat([rets['MORE'].rename('MORE'), fac, rets['XIC']], axis=1, join='inner').dropna()
X = np.column_stack([np.ones(len(df)), df['WTI'], df['Gold'], df['XIC']])
beta, *_ = np.linalg.lstsq(X, df['MORE'].values, rcond=None)
resid = df['MORE'].values - X @ beta
r2 = 1 - float(resid @ resid) / float(((df['MORE'].values - df['MORE'].mean()) ** 2).sum())
print('R squared:', round(float(r2), 3))
print('Betas: alpha', round(float(beta[0]), 4), '| WTI', round(float(beta[1]), 3),
      '| Gold', round(float(beta[2]), 3), '| XIC', round(float(beta[3]), 3))
alpha_ann = float(beta[0]) * 52
for label, o, g, t in [('Base', 0.05, 0.05, 0.07),
                       ('Energy rally', 0.20, 0.00, 0.10),
                       ('Gold rally', 0.00, 0.20, 0.05),
                       ('Both fall', -0.20, -0.10, -0.10),
                       ('Oil crash, gold safety', -0.30, 0.10, -0.05)]:
    print(label, 'implied MORE move:',
          round(float((alpha_ann + beta[1] * o + beta[2] * g + beta[3] * t) * 100), 2), '%')

## Method and limits

Weekly Friday prices in CAD from Yahoo Finance, adjusted for dividends, 20 September 2024 to 13 November 2026, 113 weeks. Annual return is the average weekly return multiplied by 52. Volatility is the standard deviation of weekly returns multiplied by the square root of 52. The Sharpe ratio uses a stated 2.0% risk free rate, not a market T-bill series. Two years is a short window, so the risk numbers are thin and the return numbers depend heavily on the gold rally of 2025 and the energy rally of 2026. The Monte Carlo uses MORE's own weekly volatility with two drifts, and the scenarios use betas from 113 weeks, so both are ranges and what-ifs, not predictions. Past performance does not predict future performance.